In [0]:

-- my first window function
SELECT
  o_custkey,
  o_orderkey,
  o_totalprice,
  ROW_NUMBER() OVER (
    PARTITION BY o_custkey       -- treat each customer as a separate group
    ORDER BY o_totalprice DESC   -- sort within the group, highest first
  ) AS rn
FROM samples.tpch.orders;

In [0]:
-- Count the orders and sum the revenue (o_totalprice) for each o_orderstatus.
SELECT 
  o_orderstatus,
  COUNT(o_orderkey) as order_count,
  SUM(o_totalprice) as total_revenue
FROM samples.tpch.orders
GROUP BY o_orderstatus
ORDER BY total_revenue DESC;

In [0]:
-- Join orders → customer → nation and return the top 5 nations by total revenue.
SELECT
  n.n_name,
  SUM(o.o_totalprice) as total_revenue
FROM samples.tpch.orders AS o
JOIN samples.tpch.customer AS c ON o.o_custkey = c.c_custkey
JOIN samples.tpch.nation AS n ON c.c_nationkey = n.n_nationkey
GROUP BY n.n_name
ORDER BY total_revenue DESC
LIMIT 5;


In [0]:
-- Return each customer's top 3 orders by price. Write it once with a CTE and once with QUALIFY.

WITH rankOrders AS(
  SELECT
    o_custkey,
    o_orderkey,
    o_totalprice, ROW_NUMBER() OVER (PARTITION BY o_custkey ORDER BY o_totalprice DESC,o_orderkey) as rn
    FROM samples.tpch.orders
)
SELECT 
  c.c_name, 
  o.o_orderkey, 
  o.o_totalprice
FROM rankOrders AS o
JOIN samples.tpch.customer AS c ON o.o_custkey = c.c_custkey
WHERE o.rn <= 3
ORDER BY c.c_name DESC;



In [0]:
-- QUALIFY
SELECT 
  c.c_name, 
  o.o_orderkey, 
  o.o_totalprice
FROM samples.tpch.orders AS o
JOIN samples.tpch.customer AS c ON o.o_custkey = c.c_custkey
QUALIFY ROW_NUMBER() OVER (PARTITION BY o.o_custkey ORDER BY o.o_totalprice DESC,o.o_orderkey) <= 3
ORDER BY c.c_name DESC;

In [0]:
-- running total of each customer's spend, ordered by o_orderdate.
SELECT
  o_custkey,
  o_orderdate,
  o_totalprice,
  SUM(o_totalprice) OVER (
  PARTITION BY o_custkey
  ORDER BY o_orderdate, o_orderkey
  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
) AS running_total
FROM samples.tpch.orders
ORDER BY o_custkey, o_orderdate
LIMIT 50;

In [0]:
-- Count the orders and sum the revenue (o_totalprice) for each o_orderstatus.
SELECT 
  o_orderstatus,
  COUNT(o_orderkey) as order_count,
  SUM(o_totalprice) as total_revenue
FROM samples.tpch.orders
GROUP BY o_orderstatus
ORDER BY total_revenue DESC;

In [0]:
-- What does your query ACTUALLY return?
SELECT COUNT(*) AS actual_rows
FROM (
  WITH rankOrders AS (
    SELECT
      o_custkey,
      o_orderkey,
      o_totalprice,
      ROW_NUMBER() OVER (PARTITION BY o_custkey ORDER BY o_totalprice DESC, o_orderkey) as rn
    FROM samples.tpch.orders
  )
  SELECT 
    c.c_name, 
    o.o_orderkey, 
    o.o_totalprice
  FROM rankOrders AS o
  JOIN samples.tpch.customer AS c ON o.o_custkey = c.c_custkey
  WHERE o.rn <= 3
) t